[Back to NLP guideline](Natural-Language-Processing.html)


## **Challenges, Risks, Ethics, and Safety**

NLP systems do not operate in an empty laboratory. They read language produced by people, learn from data shaped by institutions and history, and influence decisions about those same people. A system can therefore be technically accurate on a benchmark while still being unreliable, unfair, unsafe, or inappropriate in its deployment context.

The central question of this chapter is not simply whether a model makes errors. Every statistical model makes errors. The more important questions are:

```text
what can fail
-> who can be affected
-> how serious is the impact
-> how can the risk be detected
-> what controls should be applied
-> who is responsible when the system fails
```

For example, an incorrect movie recommendation is usually a low-impact error. The same type of unsupported generation in a medical summary can influence treatment and become a high-impact safety failure. Risk depends on both model behavior and context of use.

This chapter treats responsible NLP as a system property. Data collection, annotation, modeling, evaluation, interface design, deployment policy, monitoring, and human oversight all contribute to the final outcome. A content filter added at the end cannot repair every problem introduced earlier in the pipeline.

| Layer | Central Question | Example Failure |
|---|---|---|
| linguistic | did the system understand the language? | sarcasm is interpreted literally |
| statistical | does the model generalize? | performance collapses in a new domain |
| factual | is the output supported and correct? | a summary invents a diagnosis |
| social | are benefits and errors distributed fairly? | one dialect receives more false toxicity flags |
| privacy | was personal data handled appropriately? | a model reproduces an email address |
| security | can an attacker manipulate the system? | an injected instruction overrides the intended task |
| operational | are failures detected and handled? | no alert is raised after a harmful distribution shift |
| governance | are responsibilities and remedies defined? | affected users cannot appeal an automated decision |

The emphasis differs from the evaluation chapter. Evaluation explains how to measure model quality. This chapter asks why certain failures matter, how they become harms, and how technical and organizational controls should respond. The next chapter will then show how these controls fit into modern RAG, tool-using, and agentic systems.

### **Understanding NLP Risks**

Risk is the possibility that an NLP system contributes to an unwanted outcome. It combines uncertainty about whether something will happen with the severity of the consequences if it does. Ethics provides principles for deciding which outcomes are acceptable, whose interests count, and what obligations developers and deployers have. Safety turns those principles into concrete engineering and operational controls.

These ideas are related but not identical:

| Concept | Primary Focus | Typical Question |
|---|---|---|
| model error | prediction differs from a target | was the label or text wrong? |
| reliability | behavior remains dependable under expected conditions | does quality remain stable across inputs and time? |
| robustness | behavior remains stable under variation or attack | does a small change cause a large failure? |
| safety | unacceptable harm is prevented or contained | can this failure injure people or systems? |
| ethics | values, rights, duties, and distribution of benefits or harms | should the system be used this way? |
| governance | authority, process, documentation, and accountability | who decides, monitors, and responds? |

#### **Technical Failures and Sociotechnical Harms**

A **technical failure** is a defect in model or system behavior. A **harm** is a negative consequence experienced by a person, group, organization, or society. The distinction matters because one failure can cause different harms in different contexts, and a harmful outcome can occur even when the model behaves as designed.

Consider a resume-ranking model that assigns lower scores to applications containing terms associated with a particular social group. The technical failure may involve biased training data or a proxy feature. The allocational harm is that qualified applicants receive fewer interviews. The organizational harm includes poor hiring decisions and loss of trust. The governance failure occurs if nobody monitors subgroup outcomes or provides an appeal route.

The relationship is often a chain rather than a single event:

```text
historical practice
-> biased data
-> learned association
-> ranking decision
-> human reliance on ranking
-> unequal access to opportunity
```

NLP is especially sociotechnical because language carries social identity, power, politeness, cultural assumptions, and context. A toxicity detector is not merely recognizing a fixed property of a sentence. It is applying a policy definition to language that may include quotation, counterspeech, reclaimed slurs, humor, or community-specific norms.

A useful analysis separates three levels:

| Level | Example | Appropriate Response |
|---|---|---|
| component failure | NER misses an address | improve data, model, or threshold |
| system failure | redaction pipeline exposes the missed address | add layered detectors and validation |
| impact failure | exposed address leads to harassment | provide incident response, support, and remedy |

Improving the model may reduce the first failure, but preventing the final harm also requires product design, access control, logging, escalation, and accountability.

#### **Stakeholders, Context, and Risk Severity**

A stakeholder is anyone who builds, operates, uses, is represented in, or is affected by the system. Direct users are only one group. A hiring manager may use a ranking model, while applicants are the people evaluated by it. A clinician may use a summarizer, while the patient is the subject of the summarized record. Bystanders can also be affected when their messages, images, or personal details enter a model input.

Before measuring risk, identify at least these roles:

| Stakeholder | Example | What May Be at Stake |
|---|---|---|
| developer | team building a classifier | correctness and maintainability |
| deployer | organization integrating the model | compliance and operational reliability |
| operator | moderator or analyst using outputs | workload and decision quality |
| direct user | person asking a chatbot | useful and safe assistance |
| decision subject | person classified or ranked | opportunity, dignity, and due process |
| data subject | person mentioned in training or input data | privacy and consent |
| affected community | group experiencing aggregate effects | representation and equitable treatment |
| auditor or regulator | independent oversight body | evidence, traceability, and accountability |

A simple prioritization model is:

$$
R = L \times I \times E
$$

Here, $R$ is the relative risk score, $L$ is the estimated likelihood of the failure, $I$ is its impact severity, and $E$ is exposure: how many people or decisions may encounter the failure. If each factor is scored from 1 to 5, the result ranges from 1 to 125. A rare but catastrophic event can still receive high priority, while a small error repeated millions of times can also become important through exposure.

This formula is a prioritization aid, not an objective measurement of human harm. Numbers can hide disagreement about whose impact is counted, whether harms are reversible, and whether affected people consented to the risk. Scores should therefore be accompanied by written reasoning.

<details>
<summary>Python Building a Small NLP Risk Register</summary>

```python
risks = [
    {
        "name": "medical summary invents a medication",
        "likelihood": 2,
        "impact": 5,
        "exposure": 4,
        "owner": "clinical NLP lead",
        "control": "source-grounded generation plus clinician approval",
    },
    {
        "name": "sentiment model fails on code-mixed reviews",
        "likelihood": 4,
        "impact": 2,
        "exposure": 3,
        "owner": "model evaluation lead",
        "control": "language-slice tests and targeted data collection",
    },
    {
        "name": "support chatbot reveals an account identifier",
        "likelihood": 2,
        "impact": 4,
        "exposure": 3,
        "owner": "privacy engineer",
        "control": "input minimization, output DLP scan, and access logging",
    },
]

# Step 1: calculate a relative score for prioritization.
for risk in risks:
    risk["score"] = (
        risk["likelihood"] * risk["impact"] * risk["exposure"]
    )

# Step 2: review the highest-priority risks first.
for risk in sorted(risks, key=lambda item: item["score"], reverse=True):
    print(f"{risk['score']:>3} | {risk['name']}")
    print(f"      owner: {risk['owner']}")
    print(f"      control: {risk['control']}")
```

</details>

Risk assessment should also consider reversibility, detectability, affected population, legal or contractual obligations, and the availability of an appeal. A low average error rate is not sufficient when one type of error is concentrated on a vulnerable group or when a single error is difficult to reverse.

### **Linguistic, Multilingual, and Domain Challenges**

Language is ambiguous, contextual, creative, and constantly changing. NLP systems convert this open-ended object into finite representations and statistical predictions. The conversion works well for many common patterns, but difficult cases reveal what the model has actually learned.

Linguistic challenges are not automatically ethical problems. They become safety or fairness problems when errors are systematic, unequally distributed, or used in consequential decisions. A dialect recognition failure in a music recommendation system is inconvenient; the same failure in emergency call transcription can be dangerous.

#### **Ambiguity, Context, and Pragmatics**

An expression is ambiguous when it supports more than one interpretation. Humans resolve ambiguity by combining grammar, surrounding text, shared knowledge, speaker goals, and social context. A model may rely on only part of this evidence.

Common forms include:

| Ambiguity | Example | What Must Be Resolved |
|---|---|---|
| lexical | `bank` | financial institution or river edge |
| syntactic | `I saw the student with the telescope` | who has the telescope? |
| referential | `Maya told Priya that she won` | who does `she` refer to? |
| scope | `Every reviewer did not agree` | none agreed or not all agreed? |
| pragmatic | `Can you open the window?` | ability question or polite request? |
| figurative | `That update was a masterpiece` | sincere praise or sarcasm? |

Context is more than nearby tokens. In dialogue, the meaning of an answer depends on previous turns, speaker roles, and conversational goals. In legal or clinical text, definitions established earlier in a document may control the meaning of later phrases. In social media, an image, link, hashtag, or community convention may be essential.

Pragmatics concerns what a speaker intends and what a listener infers beyond literal semantics. Irony, indirect requests, presupposition, implicature, politeness, and humor are difficult because the same words can perform different social actions. A model that labels `Great, another cancelled train` as positive because of the word `great` has recognized lexical sentiment but missed pragmatic sentiment.

The practical response is not simply to use a larger model. Systems can preserve more conversational context, include speaker and domain metadata when appropriate, test contrastive examples, and allow uncertainty or abstention. Sensitive applications should avoid treating a single decontextualized sentence as a complete representation of a person's intent.

#### **Compositional and Long-Range Reasoning**

Compositionality means that the meaning of a larger expression depends on its parts and how they are combined. NLP models often learn useful lexical associations but struggle when a familiar word appears inside a new logical structure.

Compare the following statements:

```text
The treatment was effective.
The treatment was not effective.
It is not true that the treatment was ineffective.
The treatment was effective for adults but not for children.
```

Keyword overlap is high, but the conclusions differ. Negation, quantifiers, conditionals, conjunction, temporal order, and nested clauses change meaning. A reliable system must represent relationships, not just count positive or negative words.

Long-range reasoning introduces another difficulty. A pronoun may refer to an entity several paragraphs earlier. A contract may define an exception on page 2 that overrides a general rule on page 30. A scientific answer may require combining evidence from multiple sections. Even when the full text fits in a context window, the model may not retrieve and integrate the correct pieces.

Useful tests include minimal pairs, where only one meaning-changing feature is altered:

```text
The patient has diabetes.      -> diagnosis present
The patient has no diabetes.   -> diagnosis absent

All claims were approved.      -> universal approval
Some claims were approved.     -> partial approval
```

Minimal pairs expose whether a prediction changes for the right reason. They are more diagnostic than a large random test set in which most examples do not contain the targeted phenomenon.

#### **Domain and Temporal Distribution Shift**

A model is usually trained on one data distribution and deployed on another. Distribution shift exists when:

$$
P_{\text{train}}(X, Y) \neq P_{\text{deploy}}(X, Y)
$$

$X$ denotes input text and $Y$ denotes the desired output. The equation says that the relationship observed during training differs from the relationship encountered in deployment.

Three forms are useful to distinguish:

| Shift | What Changes | NLP Example |
|---|---|---|
| covariate shift | $P(X)$ changes | formal news becomes short social posts |
| label shift | $P(Y)$ changes | the frequency of fraud reports increases |
| concept shift | $P(Y \mid X)$ changes | a phrase acquires a new meaning over time |

Domain shift occurs across sources such as news, medicine, law, education, customer support, or different organizations. Temporal shift occurs because language and the world change. New products, public figures, events, slang, policies, and disease names appear. Labeling policy can change as well: a moderation category considered acceptable last year may become disallowed after a policy revision.

Random train-test splits often underestimate this problem because both sets contain text from the same period and source. More realistic evaluation may hold out an organization, time period, author group, genre, or topic. Deployment monitoring should then compare incoming data and slice-level performance with the conditions documented during development.

#### **Multilinguality, Dialects, and Code-Switching**

Multilingual NLP is not simply English NLP repeated with a different vocabulary. Languages differ in morphology, word order, scripts, writing conventions, available resources, and cultural assumptions. Tokenizers and models trained on imbalanced web data may allocate much better representations to high-resource languages.

Dialect variation is systematic language variation, not corrupted standard language. Treating dialect features as spelling errors can erase identity and create unequal quality. In moderation, a classifier may over-associate vocabulary used by a community with toxicity. In speech recognition, accent and dialect errors can make a downstream assistant appear unresponsive or less capable for certain users.

**Code-switching** occurs when speakers alternate languages or language varieties within one conversation or sentence:

```text
The service was fast, pero la comida no estaba buena.
```

A monolingual sentiment system may understand only the positive English clause and miss the negative Spanish clause. Code-switching can also involve transliteration, borrowed words, mixed scripts, and community-specific grammar. Language identification at the document level is therefore often too coarse.

Strong multilingual design uses language- and dialect-specific evaluation slices, native-speaker review, appropriate tokenization, and task data from the actual deployment community. Machine translation can help bootstrap a system, but translated evaluation data does not fully reproduce naturally produced language.

#### **Low-Resource Languages and Cultural Representation**

A low-resource language has limited digital text, annotations, tools, benchmarks, or compute investment relative to the task. Resource scarcity is not a property of the language itself. It often reflects historical, economic, and institutional choices about what is digitized and funded.

Common consequences include:

| Constraint | Model Consequence | Responsible Response |
|---|---|---|
| little unlabeled text | weak pretrained representations | community-approved data collection and multilingual transfer |
| few labels | unstable supervised models | active learning, weak supervision, and careful uncertainty reporting |
| no benchmark | quality is difficult to compare | create representative evaluation with local experts |
| limited written standard | spelling and grammar vary | avoid enforcing one variety as universally correct |
| imported labels | categories mismatch local concepts | redesign the task with community participation |

Cultural representation goes beyond translation quality. Emotion categories, politeness, offensiveness, identity terms, and conversational expectations vary across communities. A label taxonomy developed in one setting may not transfer. For affective computing, for example, the same expression may communicate distress, respect, humor, or indirect disagreement depending on cultural context.

<details>
<summary>Python Evaluating Linguistic and Domain Slices</summary>

```python
from collections import defaultdict

examples = [
    {"slice": "standard", "gold": "positive", "pred": "positive"},
    {"slice": "standard", "gold": "negative", "pred": "negative"},
    {"slice": "negation", "gold": "negative", "pred": "positive"},
    {"slice": "negation", "gold": "positive", "pred": "positive"},
    {"slice": "code_switch", "gold": "negative", "pred": "positive"},
    {"slice": "code_switch", "gold": "negative", "pred": "negative"},
    {"slice": "new_domain", "gold": "neutral", "pred": "positive"},
    {"slice": "new_domain", "gold": "neutral", "pred": "neutral"},
]

# Step 1: count correct and total predictions within each meaningful slice.
stats = defaultdict(lambda: {"correct": 0, "total": 0})
for item in examples:
    group = stats[item["slice"]]
    group["total"] += 1
    group["correct"] += int(item["gold"] == item["pred"])

# Step 2: report slice performance instead of relying only on one average.
for slice_name, values in stats.items():
    accuracy = values["correct"] / values["total"]
    print(f"{slice_name:>12}: accuracy={accuracy:.2f}, n={values['total']}")

# Step 3: production tests should add confidence intervals and enough examples
# for every slice. A tiny slice is useful for diagnosis, not for a final claim.
```

</details>

The overall lesson is that linguistic coverage must be designed and measured. A model that performs well on average may still be unreliable for negation, long documents, a new domain, a dialect, or a low-resource language. These are not edge cases when they describe the real users of the system.

### **Factuality, Faithfulness, and Hallucination**

Fluent language is not evidence of truth. A generative model is trained to produce plausible continuations, so it can generate a grammatically polished statement without possessing reliable evidence for that statement. This gap between linguistic plausibility and epistemic support is one of the most important limitations of modern NLP.

Three terms should be separated:

| Property | Reference Used for Judgment | Question |
|---|---|---|
| factuality | external world or authoritative knowledge | is the statement true? |
| faithfulness | supplied source, data, instruction, or evidence | is the statement supported by the input? |
| consistency | other parts of the same output or conversation | does the output contradict itself? |

A statement can be factual but unfaithful. Suppose a source article discusses a company's revenue but never mentions its founding year. A summary that adds the correct founding year may be factually true, yet it is not supported by the source and is therefore unfaithful as a summary. In a source-grounded task, this addition may still be unacceptable.

#### **Types and Causes of Hallucination**

Hallucination is generated content that is unsupported, contradictory, fabricated, or otherwise disconnected from the relevant evidence. The exact definition depends on the task because the relevant evidence changes.

In source-grounded generation, two traditional categories are useful:

| Type | Definition | Example |
|---|---|---|
| intrinsic hallucination | output contradicts the provided source | source says `approved in 2019`; summary says `approved in 2021` |
| extrinsic hallucination | output adds a claim not verifiable from the source | summary adds a clinical trial not mentioned in the article |

Open-domain assistants introduce additional forms such as fabricated entities, nonexistent citations, incorrect dates, false attribution, invalid reasoning, and uncertainty disguised as confidence. These forms can overlap. A fake paper title is both a factual error and a fabricated citation.

> Image: an overview of causes, detection, benchmarks, and mitigation for LLM hallucination.
>
> ![Hallucination causes, detection, and mitigation](assets/llm-hallucination-landscape.png){width=95%}
>
> Source: [Huang et al. - A Survey on Hallucination in Large Language Models](https://arxiv.org/abs/2311.05232)

Hallucination can enter at several stages:

| Stage | Mechanism | Example |
|---|---|---|
| data | sources contain errors or source-target pairs diverge | training summary includes facts absent from its article |
| representation | the model compresses incomplete or conflicting knowledge | similar entities are blended together |
| training | the objective rewards plausible text rather than verified truth | likely wording receives high probability |
| alignment | preference data rewards confident helpfulness | the model guesses instead of admitting uncertainty |
| decoding | sampling selects an unsupported continuation | temperature amplifies a low-probability false claim |
| retrieval | irrelevant or stale evidence enters the context | answer is grounded in the wrong document |
| orchestration | tool output is omitted, truncated, or misinterpreted | final answer contradicts the calculator or database |

The problem should therefore not be attributed to decoding alone. Lower temperature can reduce randomness, but a deterministic model can confidently reproduce incorrect knowledge. Likewise, retrieval can supply evidence, but it cannot guarantee that the evidence is relevant, trustworthy, or correctly used.

#### **Factuality vs Faithfulness**

Factuality compares an output with the world. Faithfulness compares it with a specified source or instruction. The distinction determines the correct evaluation method.

Consider a hospital discharge note:

```text
Source: The patient takes 5 mg of Drug A each morning.

Output A: The patient takes 10 mg of Drug A each morning.
Output B: The patient also takes Drug B in the evening.
Output C: The patient takes 5 mg of Drug A each morning.
```

Output A directly contradicts the source and is intrinsically unfaithful. Output B is unsupported by the source; it might happen to be true in the world, but the summarizer has no evidence for it. Output C is source-faithful, although a separate issue could arise if the source record itself is outdated.

This creates two verification layers:

```text
output vs supplied evidence -> faithfulness check
output vs reliable external evidence -> factuality check
```

For summarization, data-to-text, and retrieval-grounded QA, faithfulness is usually the first requirement. For open-domain question answering, external factuality becomes central. For creative writing, invention may be intended, but consistency with user constraints still matters.

| Task | Primary Grounding Source | Main Failure |
|---|---|---|
| summarization | source document | unsupported or contradictory summary |
| data-to-text | table or database | wrong value, relation, or entity |
| machine translation | source sentence | added, omitted, or altered meaning |
| retrieval-grounded QA | retrieved evidence | answer not entailed by evidence |
| open-domain QA | reliable world knowledge | false or fabricated claim |
| dialogue | conversation state and tool results | inconsistent persona, state, or action |

#### **Grounding and Uncertainty**

Grounding connects a generated claim to evidence that can support or refute it. Evidence may come from an input document, structured database, retrieved passage, verified tool result, or human expert. A grounded system should preserve provenance so the evidence can be inspected.

Uncertainty describes what the system does not know or cannot reliably infer. Token probability is not the same as factual confidence. A model can assign high probability to a common misconception and low probability to a rare but correct technical term.

One basic uncertainty measure for a probability distribution is entropy:

$$
H(p) = -\sum_{i=1}^{K} p_i \log p_i
$$

$K$ is the number of possible outputs or classes, and $p_i$ is the probability assigned to option $i$. Entropy is low when probability is concentrated on one option and high when it is spread across many options. In classification this can indicate uncertainty, but only if probabilities are reasonably calibrated. In generation, token entropy measures uncertainty about the next token, not whether the complete claim is true.

Practical uncertainty mechanisms include:

| Mechanism | Purpose | Limitation |
|---|---|---|
| calibrated confidence | align reported confidence with observed correctness | calibration can drift by domain |
| abstention | decline when evidence is insufficient | too much abstention reduces usefulness |
| conformal prediction | produce prediction sets with empirical coverage | assumptions may weaken under shift |
| evidence citation | expose the grounding source | a citation can be irrelevant or misread |
| multiple samples | detect unstable answers | repeated models can share the same error |
| human review | escalate consequential uncertainty | reviewers have limited time and may over-trust the model |

A safe system should know when to switch from answer generation to clarification, retrieval, tool use, or human escalation. The interface should communicate uncertainty without pretending that a numeric confidence score is more precise than the evidence allows.

#### **Detection and Mitigation**

Hallucination detection is stronger when it checks atomic claims rather than scoring an entire paragraph as one unit. A practical pipeline is:

```text
generated response
-> split into checkable claims
-> retrieve or identify evidence for each claim
-> classify evidence as support, contradiction, or insufficient
-> aggregate claim results
-> revise, abstain, or escalate
```

Different tools answer different questions:

| Method | Signal | Best Use | Main Limitation |
|---|---|---|---|
| lexical overlap | shared words or phrases | quick source coverage check | misses paraphrases and logical errors |
| NLI model | entailment or contradiction | source-grounded claims | inherits NLI model errors |
| QA-based checking | answer consistency | summaries and long documents | question generation can omit key claims |
| retrieval plus verification | external evidence | open-domain factuality | retrieval quality bounds verification |
| knowledge-base lookup | structured facts | entities, dates, and relations | coverage is incomplete |
| human fact-checking | expert judgment | high-impact or ambiguous claims | slow and expensive |

Mitigation can occur before, during, or after generation. Data filtering and corrected supervision improve the knowledge source. Retrieval and tools provide external evidence. Constrained generation and decoding reduce unsupported variation. Post-generation verification can remove or flag claims. High-impact systems should combine several layers.

<details>
<summary>Python Claim-Level Evidence Checking Workflow</summary>

```python
import re

source = (
    "The trial enrolled 120 adults. Participants received Drug A for "
    "eight weeks. The report did not evaluate children."
)

generated_claims = [
    "The trial enrolled 120 adults.",
    "Participants received Drug A for eight weeks.",
    "The treatment was proven safe for children.",
]

def content_words(text):
    """A tiny lexical baseline, not a production fact checker."""
    tokens = re.findall(r"[a-z0-9]+", text.lower())
    stopwords = {"the", "a", "an", "for", "was", "were", "is", "of"}
    return {token for token in tokens if token not in stopwords}

def lexical_support(claim, evidence):
    # Step 1: compare the checkable words in the claim with source words.
    claim_terms = content_words(claim)
    evidence_terms = content_words(evidence)
    return len(claim_terms & evidence_terms) / max(len(claim_terms), 1)

review_queue = []

for claim in generated_claims:
    # Step 2: score each claim independently so one supported sentence
    # does not hide another unsupported sentence.
    score = lexical_support(claim, source)

    # Step 3: route weakly supported claims to a stronger NLI checker,
    # external retrieval, or a human reviewer.
    status = "provisionally supported" if score >= 0.75 else "needs review"
    print(f"{status:>23} | overlap={score:.2f} | {claim}")

    if status == "needs review":
        review_queue.append(claim)

print("review queue:", review_queue)
```

</details>

The lexical score intentionally demonstrates the workflow rather than a complete solution. The unsupported child-safety claim shares some words with the source and could fool a naive overlap metric. A production checker needs semantic entailment, contradiction detection, provenance, and domain expertise.

#### **Limits of Retrieval and Self-Verification**

Retrieval-Augmented Generation can reduce dependence on parametric memory, but it moves part of the problem into retrieval and evidence use. The system can retrieve the wrong passage, miss the relevant passage, use stale content, select an untrustworthy source, or generate a claim not entailed by the retrieved text.

Common RAG failure patterns include:

| Failure | What Happened | Control |
|---|---|---|
| retrieval miss | correct evidence never enters the context | hybrid retrieval, query rewriting, recall tests |
| ranking error | relevant evidence is ranked too low | reranking and document-level evaluation |
| context overload | too many passages hide the key evidence | context selection and compression |
| source conflict | retrieved documents disagree | provenance, recency, authority ranking, explicit conflict reporting |
| citation mismatch | citation exists but does not support the claim | claim-citation entailment checks |
| generation drift | model ignores or alters evidence | grounded prompts, constrained output, post-checking |

Self-verification asks the same or another model to critique an answer. It can catch arithmetic mistakes, contradictions, and unstable reasoning, but it is not independent evidence. The verifier may share the generator's training data, misconceptions, or blind spots. Repeated confidence is not proof.

A practical trust order is:

```text
authoritative external evidence
> independently executed tool result
> independently trained verifier
> same-model critique
> unsupported model confidence
```

The right mitigation depends on the task. Creative generation may tolerate invention. Scientific summarization should require strict source faithfulness. Medical, legal, and financial assistance should combine authoritative evidence, bounded scope, explicit uncertainty, and qualified human review.

### **Bias, Fairness, and Inclusion**

Bias in NLP refers to systematic patterns that can distort representation, quality, or outcomes. Not every statistical difference is automatically unfair, and fairness is not one universal number. The analysis must identify what behavior is harmful, to whom, in which context, and according to which normative principle.

Language models learn from human-produced data. That data contains valuable cultural knowledge as well as stereotypes, exclusions, unequal visibility, institutional practices, and annotation judgments. A model can reproduce these patterns, amplify them, or create new interactions when deployed at scale.

#### **Sources of Bias Across the NLP Pipeline**

Bias can enter before a model is trained and after it is deployed. Focusing only on model parameters misses problem formulation, data collection, annotation, metric selection, user interface, and feedback loops.

> Image: NIST distinguishes systemic, human, and statistical or computational sources of bias.
>
> ![Categories of AI bias](assets/nist-bias-categories.png){width=80%}
>
> Source: [NIST SP 1270 - Towards a Standard for Identifying and Managing Bias in Artificial Intelligence](https://doi.org/10.6028/NIST.SP.1270)

In an NLP pipeline, sources include:

| Pipeline Stage | Possible Bias | Example |
|---|---|---|
| task definition | wrong target or proxy | predicting `culture fit` from writing style |
| data collection | population and selection bias | only highly active users are represented |
| labeling | annotator and policy bias | dialectal language is labeled more toxic |
| preprocessing | exclusion bias | names in a non-Latin script are removed |
| representation | coverage imbalance | tokenizer fragments one language much more heavily |
| training objective | majority patterns dominate | rare identity contexts receive little learning signal |
| evaluation | benchmark bias | test set contains only standard written English |
| deployment | automation and interaction bias | users treat a score as an objective fact |
| feedback loop | past decisions become future labels | previous ranking determines what data is observed |

Removing an explicit protected attribute does not necessarily remove bias. Names, locations, schools, vocabulary, and writing patterns can act as proxies. In some cases, protected attributes are also needed for auditing subgroup performance. The responsible question is how data is governed and used, not simply whether a column exists.

#### **Representational and Allocational Harm**

**Representational harm** concerns how people and groups are depicted, recognized, or made visible. **Allocational harm** concerns how opportunities, resources, or burdens are distributed.

| Harm Type | NLP Example | Potential Consequence |
|---|---|---|
| stereotyping | occupations are associated with one gender | social stereotypes are reinforced |
| denigration | identity language is generated with insults | dignity and psychological harm |
| erasure | a language or identity is consistently omitted | users become invisible to the system |
| quality-of-service disparity | ASR error is higher for one accent | unequal access to a service |
| allocational harm | resume rank differs by proxy identity cues | unequal employment opportunity |
| surveillance burden | one community is flagged more often | disproportionate review or enforcement |

The same system can produce several harms. A toxicity classifier that over-flags a dialect creates a quality disparity, suppresses community expression, and may allocate moderation penalties unevenly. Measuring only average accuracy would hide all three.

Context also determines harm. A stereotype in a fictional brainstorming tool differs from the same association inside a hiring recommendation. This does not make the first case harmless; it means the likelihood, exposure, and consequences must be analyzed separately.

#### **Group, Individual, and Intersectional Fairness**

Group fairness compares outcomes or error rates across defined groups. Individual fairness asks whether similar people are treated similarly. Intersectional fairness examines combinations of attributes, such as language, gender, age, and disability, because aggregate categories can hide concentrated failures.

Suppose $A$ is a group attribute, $Y$ is the true label, and $\hat{Y}$ is the model decision. Common group metrics include:

**Demographic parity difference** compares positive decision rates:

$$
\Delta_{DP} = P(\hat{Y}=1 \mid A=a) - P(\hat{Y}=1 \mid A=b)
$$

$P(\hat{Y}=1 \mid A=a)$ is the proportion of group $a$ receiving a positive prediction. A value near zero indicates similar selection rates. However, demographic parity does not consider whether the true label distribution or label quality differs.

**Equal opportunity difference** compares true-positive rates:

$$
\Delta_{EOpp} = P(\hat{Y}=1 \mid Y=1, A=a) - P(\hat{Y}=1 \mid Y=1, A=b)
$$

This asks whether qualified or truly positive cases are recognized at similar rates. In a beneficial-resource setting, a lower true-positive rate means one group misses more deserved positive outcomes.

**Equalized odds** requires both true-positive and false-positive rates to be similar. It considers access to positive outcomes and exposure to incorrect positive decisions. Which error matters most depends on the application.

Fairness definitions can conflict. If groups have different base rates and predictions are imperfect, calibration, equalized odds, and equal positive rates may not all be achievable at once. Selecting a metric is therefore an ethical and policy decision, not merely a mathematical convenience.

Intersectional analysis is essential because a model can appear fair for gender and fair for language separately while failing for women who use a particular low-resource language. Slice sizes and uncertainty should always be reported so a noisy estimate is not mistaken for evidence of parity.

#### **Measuring and Mitigating Bias**

Bias evaluation should combine quantitative slices, controlled tests, qualitative review, and feedback from affected communities. A benchmark may show a gap without explaining its cause, while interviews may reveal harms absent from the benchmark.

<details>
<summary>Python Computing Subgroup Error and Fairness Gaps</summary>

```python
from collections import defaultdict

records = [
    # gold=1 means the message truly requires urgent support.
    {"group": "A", "gold": 1, "pred": 1},
    {"group": "A", "gold": 1, "pred": 1},
    {"group": "A", "gold": 0, "pred": 1},
    {"group": "A", "gold": 0, "pred": 0},
    {"group": "B", "gold": 1, "pred": 0},
    {"group": "B", "gold": 1, "pred": 1},
    {"group": "B", "gold": 0, "pred": 0},
    {"group": "B", "gold": 0, "pred": 0},
]

counts = defaultdict(lambda: {"tp": 0, "fp": 0, "tn": 0, "fn": 0})

# Step 1: build a confusion matrix for each group.
for row in records:
    if row["gold"] == 1 and row["pred"] == 1:
        counts[row["group"]]["tp"] += 1
    elif row["gold"] == 0 and row["pred"] == 1:
        counts[row["group"]]["fp"] += 1
    elif row["gold"] == 0 and row["pred"] == 0:
        counts[row["group"]]["tn"] += 1
    else:
        counts[row["group"]]["fn"] += 1

rates = {}
for group, c in counts.items():
    # Step 2: calculate rates tied to meaningful error types.
    tpr = c["tp"] / max(c["tp"] + c["fn"], 1)
    fpr = c["fp"] / max(c["fp"] + c["tn"], 1)
    selection = (c["tp"] + c["fp"]) / sum(c.values())
    rates[group] = {"tpr": tpr, "fpr": fpr, "selection": selection}
    print(group, rates[group], "n=", sum(c.values()))

# Step 3: report signed gaps and inspect the underlying rates.
equal_opportunity_gap = rates["A"]["tpr"] - rates["B"]["tpr"]
demographic_parity_gap = rates["A"]["selection"] - rates["B"]["selection"]

print("equal opportunity gap:", equal_opportunity_gap)
print("demographic parity gap:", demographic_parity_gap)
```

</details>

Mitigation can occur at several stages:

| Stage | Technique | Benefit | Risk or Limitation |
|---|---|---|---|
| problem design | replace an inappropriate proxy or automate less | addresses the root decision | may require organizational change |
| data | improve coverage, documentation, and labels | reduces representation gaps | collection can create privacy burdens |
| training | reweighting, resampling, adversarial objectives | changes learned behavior | may trade off metrics or overfit group labels |
| inference | group-aware thresholds where lawful and justified | can target error-rate gaps | policy and legal implications vary |
| interface | show uncertainty and permit correction | reduces over-reliance | users may still ignore warnings |
| operations | monitoring, appeal, and remediation | catches deployment harms | requires ownership and resources |

Data balancing alone is not a complete solution. Equal sample counts do not guarantee equal linguistic diversity, label quality, or social meaning. Synthetic examples can reproduce the generator's stereotypes. Debiasing one benchmark may suppress visible signals without changing the underlying association.

#### **Fairness Trade-Offs and Limitations**

Fairness work contains unavoidable choices. Improving recall for one group may increase false positives. Removing identity terms may reduce measured stereotype associations while making a system unable to discuss discrimination. A single global threshold may produce unequal error rates, while group-specific thresholds may be inappropriate or prohibited in some contexts.

Responsible decisions should document:

```text
the affected outcome
-> the groups and intersections examined
-> why a fairness definition was selected
-> uncertainty in each estimate
-> expected trade-offs
-> who participated in the decision
-> how affected people can contest outcomes
```

Fairness is not achieved once and then frozen. Language, populations, policies, and usage change. A model that met a fairness target during development can drift after deployment. Continuous slice monitoring and channels for lived experience are therefore part of fairness, not optional additions.

### **Privacy, Copyright, and Data Governance**

NLP systems process language that frequently contains personal, confidential, copyrighted, or operationally sensitive information. Privacy is not limited to hiding names. A text may reveal health conditions, political views, relationships, location, workplace events, or identity through combinations of details.

Data governance defines how data is collected, justified, documented, accessed, retained, shared, corrected, and deleted. Privacy controls protect people within that lifecycle. Copyright and licensing address separate questions about rights to use and reproduce content. These concerns overlap, but satisfying one does not automatically satisfy the others.

#### **Personally Identifiable and Sensitive Information**

Personally identifiable information, or PII, is information that can identify a person directly or in combination with other information. Direct identifiers include names, account numbers, email addresses, and phone numbers. Quasi-identifiers such as age, suburb, occupation, rare disease, and event date may identify someone when combined.

Sensitive information can include:

| Category | Text Example | Potential Risk |
|---|---|---|
| identity | name, passport number, student ID | impersonation or identity theft |
| contact | email, phone, address | unwanted contact or harassment |
| health | diagnosis, medication, therapy notes | discrimination or personal harm |
| financial | bank details, debt, salary | fraud or economic harm |
| location | precise movement or workplace | stalking or physical risk |
| beliefs and affiliations | religion, politics, union membership | profiling or discrimination |
| private communications | support chat, legal email, diary | breach of confidence |

Two privacy failures are easy to confuse. **Disclosure** occurs when the system exposes information it was given or stored. **Sensitive inference** occurs when the system derives a private attribute from apparently less sensitive signals. Removing a medical condition from a profile does not prevent a model from inferring it from medication names or community membership.

Data minimization is the first defense: collect only what the task genuinely needs, send only the necessary fields to the model, and retain them only as long as justified. Redaction is helpful but imperfect. Regular expressions can catch standardized identifiers, while names, relationships, and contextual secrets require entity models, rules, and sometimes human review.

#### **Memorization and Training Data Extraction**

Memorization means that a model's parameters retain information about particular training examples. Regurgitation means that the model emits recognizable training content. Extraction is an adversarial or investigative process that intentionally searches for memorized content.

These are related but different:

```text
training example influences parameters
-> model memorizes some distinctive sequence
-> a prompt elicits the sequence
-> the output reveals private or copyrighted content
```

Rare, duplicated, highly distinctive, or predictable strings can be especially vulnerable. Large-scale training does not guarantee that individual examples disappear into an anonymous average.

> Image: the extraction workflow used to identify memorized GPT-2 training sequences.
>
> ![Training data extraction attack workflow](assets/training-data-extraction.png){width=95%}
>
> Source: [Carlini et al. - Extracting Training Data from Large Language Models](https://www.usenix.org/conference/usenixsecurity21/presentation/carlini-extracting)

The figure separates attack and evaluation. Candidate generations are produced and ranked using signals that may indicate memorization. Duplicates are removed, top candidates are inspected, and matches are checked against the original data. The important lesson is that ordinary generation quality and privacy leakage are different properties. A model can generalize well while still exposing a small number of rare examples.

Controls include deduplicating training data, removing secrets and PII, limiting overfitting, privacy testing, restricting access, rate limiting, output scanning, and privacy-preserving training. No single control guarantees that all memorized content has been removed.

#### **Consent, Provenance, Licensing, and Copyright**

Consent asks whether people knowingly agreed to a particular use of their data. Provenance records where data came from and how it was transformed. Licensing specifies contractual permissions and restrictions. Copyright concerns rights in protected expression. A dataset can be publicly accessible while still carrying privacy, licensing, or ethical constraints.

These questions should be documented separately:

| Question | Example Evidence |
|---|---|
| where did the text come from? | URL, repository, vendor, collection method |
| who created or is represented in it? | author, speakers, data subjects, communities |
| what permission applies? | consent record, license, contract, organizational authority |
| what transformation occurred? | filtering, translation, annotation, deduplication |
| what uses are excluded? | commercial use, redistribution, sensitive inference |
| can the data be corrected or removed? | deletion workflow and identity verification |

Training use, retrieval use, and output reproduction create different issues. A RAG system may store copyrighted documents under an authorized internal license but still expose too much source text to an unauthorized user. A model may not reproduce a document verbatim yet still have been trained on data whose provenance is unknown. Governance must follow the complete system, not only the model weights.

Legal requirements vary by jurisdiction, contract, sector, and use case. An engineering notebook should therefore identify the relevant questions and evidence rather than present one universal legal conclusion. High-impact projects need qualified legal and privacy review alongside technical controls.

#### **Retention, Deletion, and Access Control**

Data often appears in more places than the training file: raw uploads, preprocessing outputs, caches, vector indexes, logs, prompt traces, model checkpoints, backups, analytics systems, and human-review tools. A deletion request is incomplete if only one copy is removed.

A useful inventory maps each data class across its lifecycle:

```text
collection
-> validation
-> preprocessing
-> training or indexing
-> inference
-> logging and review
-> archival or deletion
```

Retention limits reduce the time during which data can be exposed. Access control should follow least privilege: users and services receive only the permissions required for their task. Encryption protects data in transit and at rest, but it does not prevent an authorized application from sending excessive data to a model or returning it to the wrong user.

Deletion from trained models is difficult. Removing a row from the source dataset does not remove its influence from an existing checkpoint. Options include retraining, machine unlearning methods, model replacement, or compensating controls such as blocking known outputs. The selected approach should match the sensitivity, evidence of memorization, feasibility, and required assurance.

#### **Privacy-Preserving NLP**

Privacy-preserving NLP combines organizational and technical methods. Anonymization attempts to make re-identification impractical, while pseudonymization replaces identifiers with controlled references. Pseudonymized data remains linkable and therefore still requires protection.

Differential privacy provides a formal guarantee that the output of a randomized mechanism changes only slightly when one person's record is added or removed. A mechanism $M$ is approximately differentially private when:

$$
P[M(D) \in S] \leq e^{\varepsilon} P[M(D') \in S] + \delta
$$

$D$ and $D'$ are neighboring datasets differing in one person's record. $M$ is the randomized training or analysis mechanism. $S$ is any possible set of outputs. The parameter $\varepsilon$ controls the privacy loss: a smaller $\varepsilon$ gives a stronger bound but usually requires more noise. $\delta$ allows a small probability that the strict bound does not hold.

The intuition is that an observer should see nearly the same distribution of outputs whether one individual participated or not. Differentially private training commonly clips per-example gradients and adds calibrated noise. It can reduce memorization risk, but it may lower utility, especially for rare patterns and underrepresented groups. Privacy and fairness effects should therefore be evaluated together.

| Method | Protects Against | Limitation |
|---|---|---|
| minimization | unnecessary collection and exposure | requires disciplined task design |
| redaction | obvious identifiers in text | context can still re-identify a person |
| access control | unauthorized internal or external use | authorized workflows can still misuse data |
| encryption | interception and storage compromise | data is visible during authorized processing |
| differential privacy | inference about individual training participation | utility and implementation trade-offs |
| federated learning | central collection of raw local data | updates can still leak information without extra protection |
| secure execution | infrastructure-level exposure | does not fix inappropriate collection or output |

<details>
<summary>Python Layered PII Detection and Redaction Baseline</summary>

```python
import re

text = (
    "Please contact Maya at maya.lee@example.org or +61 412 345 678. "
    "Her support ticket is TKT-48291."
)

patterns = {
    "EMAIL": re.compile(r"\b[A-Za-z0-9._%+-]+@[A-Za-z0-9.-]+\.[A-Za-z]{2,}\b"),
    "PHONE": re.compile(r"\+?\d[\d\s()-]{7,}\d"),
    "TICKET": re.compile(r"\bTKT-\d{4,}\b"),
}

findings = []
redacted = text

# Step 1: detect standardized identifiers with auditable rules.
for label, pattern in patterns.items():
    for match in pattern.finditer(text):
        findings.append(
            {"type": label, "value": match.group(), "span": match.span()}
        )

# Step 2: replace matches from right to left so character positions stay valid.
for item in sorted(findings, key=lambda value: value["span"][0], reverse=True):
    start, end = item["span"]
    redacted = redacted[:start] + f"[{item['type']}]" + redacted[end:]

print("redacted:", redacted)
print("audit findings:", findings)

# Step 3: a production pipeline should add named-entity recognition,
# domain-specific rules, confidence thresholds, and human review for
# high-sensitivity documents. Do not log raw values unnecessarily.
```

</details>

The code is a transparent baseline, not anonymization. It misses the person's name and any contextual combination that could identify her. Layered detection, minimization, retention controls, and review are necessary when consequences are serious.

### **Robustness and Security**

Robustness asks whether an NLP system remains dependable when inputs or environments vary. Security asks whether an adversary can intentionally exploit the system, its data, its interfaces, or its dependencies. A spelling mistake is usually a benign robustness challenge; a carefully selected trigger designed to force a target prediction is a security challenge.

Both require a threat model. A model cannot be declared universally robust or secure. The claim must specify the expected variation, attacker capabilities, protected assets, acceptable failure rate, and deployment environment.

#### **Noisy Inputs and Distribution Shift**

Real inputs differ from curated benchmarks. Text may contain typos, OCR artifacts, ASR errors, emojis, abbreviations, copied formatting, unusual Unicode, incomplete sentences, or mixed languages. Meaning-preserving paraphrases can also change model predictions even though a human sees the same intent.

Robustness evaluation should distinguish meaning-preserving changes from meaning-changing changes:

| Perturbation | Expected Behavior |
|---|---|
| punctuation or whitespace change | prediction should normally remain stable |
| common misspelling | prediction should normally remain stable |
| synonym or paraphrase | semantics and prediction should remain similar |
| inserted negation | prediction may need to change |
| changed entity or number | factual output may need to change |
| domain-specific use of a word | system should follow domain meaning |

Blind invariance is dangerous. A classifier should ignore harmless spacing but respond to `not`, changed dosage, or a new date. Robustness tests must encode which transformations preserve the task label.

Domain shift, discussed earlier, is a system-level robustness issue. Monitoring input statistics alone is insufficient because two text distributions can look similar while the relationship between text and label changes. Periodic labeled samples, user feedback, and domain-expert review are needed.

#### **Adversarial Examples and Evasion**

An adversarial example is an input intentionally modified to cause a model failure while preserving enough meaning or plausibility to pass unnoticed. Text is discrete, so attacks may alter characters, words, syntax, prompts, formatting, or surrounding context.

Universal adversarial triggers are short input sequences optimized to produce a target behavior across many examples rather than one specific example.

> Image: gradient-guided search for a universal trigger that pushes positive reviews toward a negative prediction.
>
> ![Universal adversarial trigger optimization](assets/universal-adversarial-trigger.png){width=65%}
>
> Source: [Wallace et al. - Universal Adversarial Triggers for Attacking and Analyzing NLP](https://aclanthology.org/D19-1221/)

Attack Success Rate is often reported as:

$$
ASR = \frac{N_{\text{successful attacks}}}{N_{\text{attempted attacks}}}
$$

$N_{\text{attempted attacks}}$ is the number of eligible examples attacked, and $N_{\text{successful attacks}}$ is the number for which the attacker achieved the defined goal. The goal must be explicit: any prediction change, a specific target label, harmful generation, or bypass of a control.

ASR should be reported with clean accuracy. A defense that prevents attacks by rejecting every input is secure only in a trivial sense and has no utility.

Defenses include adversarial training, normalization, ensemble checks, rate limits, anomaly detection, semantic consistency tests, and robust system design. Adaptive evaluation is essential because attackers can change their strategy after seeing the defense.

#### **Data Poisoning and Backdoors**

Data poisoning modifies training, fine-tuning, preference, retrieval, or feedback data to influence later behavior. A backdoor is a hidden behavior activated by a trigger while ordinary inputs appear normal.

Examples include:

| Attack Surface | Example |
|---|---|
| pretraining data | repeated false association is injected into web text |
| fine-tuning data | mislabeled examples shift a classifier boundary |
| preference data | harmful behavior is repeatedly ranked as preferred |
| retrieval corpus | malicious document is indexed as authoritative evidence |
| user feedback | coordinated ratings push a system toward a target response |
| model artifact | a modified checkpoint contains trigger behavior |

Poisoning is difficult to detect because large datasets already contain noise. Useful controls include provenance, trusted data boundaries, deduplication, anomaly detection, signed artifacts, reproducible builds, holdout canaries, influence analysis, and post-training behavioral tests.

A clean validation score does not rule out a backdoor. The model may behave correctly on ordinary data and fail only when the trigger appears. Security tests must include trigger search, rare-pattern slices, and supply-chain review.

#### **Prompt Injection and Jailbreaking**

Prompt injection occurs when untrusted content contains instructions that the model treats as authoritative. In a RAG system, the malicious instruction may appear inside a retrieved document. In a tool-using system, it may arrive through a webpage, email, file, or tool result. This is **indirect prompt injection** because the attacker does not need to control the user's direct prompt.

A jailbreak attempts to bypass behavioral restrictions, often through role-play, encoding, multi-turn manipulation, or instruction conflict. Prompt injection and jailbreaking overlap, but prompt injection is fundamentally an integrity problem: data is confused with control instructions.

The critical engineering principle is:

```text
natural-language instructions are not a security boundary
```

Telling a model to ignore malicious instructions is useful guidance but not access control. The surrounding application must validate tool arguments, restrict permissions, separate trusted and untrusted content, require confirmation for consequential actions, and verify outputs before execution.

Controls include:

| Control | Purpose |
|---|---|
| input provenance labels | distinguish user, developer, retrieved, and tool content |
| least-privilege tools | limit what a compromised model can do |
| schema validation | reject malformed or excessive tool arguments |
| allowlists | constrain destinations, commands, files, or actions |
| human confirmation | approve irreversible or high-impact actions |
| output encoding | prevent generated content from becoming executable markup or code |
| sandboxing | contain tool execution and data access |
| audit logs | reconstruct attempted and completed actions |

The detailed architecture belongs in the modern systems chapter. Here, the key point is that an LLM is an untrusted decision component inside a larger security design.

#### **Threat Modeling and Red-Teaming**

Threat modeling identifies what must be protected, who may attack it, how they can interact with the system, and what controls reduce risk. It should be performed before deployment and updated as the system gains new data sources, tools, users, or capabilities.

A practical NLP threat model records:

```text
assets -> private data, model behavior, tool permissions, service availability
actors -> ordinary users, malicious users, insiders, compromised suppliers
surfaces -> prompts, uploads, APIs, retrieval corpus, feedback, logs, checkpoints
goals -> extraction, manipulation, bypass, impersonation, disruption
controls -> prevention, detection, containment, recovery
```

Red-teaming is an organized attempt to discover realistic failures. It should include domain experts and people familiar with affected communities, not only security engineers. Tests should cover misuse, ambiguous edge cases, multilingual behavior, chained interactions, and attacks against the complete application.

<details>
<summary>Python Meaning-Preserving Robustness Test Harness</summary>

```python
def perturbations(text):
    """Transformations expected to preserve sentiment for this test."""
    return {
        "original": text,
        "extra_spaces": text.replace(" ", "  "),
        "lowercase": text.lower(),
        "punctuation": text.rstrip(".!?") + "!!!",
        "common_typo": text.replace("excellent", "excelllent"),
    }

def mock_sentiment_model(text):
    # Placeholder for model.predict(text). This deliberately fragile rule
    # lets the test reveal sensitivity to a common typo.
    return "positive" if "excellent" in text.lower() else "neutral"

base_text = "The support team provided excellent service."
base_prediction = mock_sentiment_model(base_text)
failures = []

# Step 1: generate only transformations whose label should remain unchanged.
for name, candidate in perturbations(base_text).items():
    prediction = mock_sentiment_model(candidate)
    stable = prediction == base_prediction
    print(f"{name:>12} | {prediction:>8} | stable={stable}")

    # Step 2: retain failed cases as regression tests.
    if not stable:
        failures.append({"test": name, "text": candidate})

# Step 3: a real harness should measure accuracy and attack success rate,
# preserve seeds and model versions, and separate benign from adversarial tests.
print("robustness failures:", failures)
```

</details>

Security findings should be prioritized by impact and exploitability, then converted into regression tests. A red-team report without remediation ownership or retesting is only a list of known problems.

| Threat | Attacker Controls | Main Property at Risk | Representative Defense |
|---|---|---|---|
| noisy input | ordinary input variation | reliability | normalization and slice testing |
| adversarial example | crafted inference input | prediction integrity | adversarial evaluation and layered detection |
| poisoning | training or index content | model and knowledge integrity | provenance and anomaly checks |
| backdoor | data or model supply chain | hidden conditional behavior | artifact verification and trigger tests |
| prompt injection | untrusted language context | instruction and tool integrity | privilege separation and validation |
| data extraction | repeated model queries | confidentiality | privacy testing, limits, and output controls |

### **Harmful Content, Misinformation, and Misuse**

Generative and analytic NLP systems can create, rank, recommend, translate, or amplify harmful language. The harm depends on content, target, intent, scale, audience, and context. A keyword list cannot reliably distinguish harassment from a news report quoting harassment, a survivor describing abuse, or counterspeech condemning it.

Safety policies therefore need operational definitions. They should state what behavior is disallowed, what is allowed with care, what needs age or domain restrictions, and when a human decision is required. Vague instructions such as `do not generate harmful content` are difficult to evaluate and enforce consistently.

#### **Toxicity, Hate Speech, and Harassment**

Toxicity is a broad label for language likely to make an interaction hostile or unsafe. Hate speech targets people based on protected or identity characteristics. Harassment directs abusive or threatening behavior toward a person or group. These categories overlap but should not be treated as synonyms.

Important contextual distinctions include:

| Context | Same Surface Term May Function As |
|---|---|
| direct attack | targeted abuse |
| quotation | evidence or reporting |
| counterspeech | condemnation of abuse |
| reclaimed language | in-group identity expression |
| educational discussion | analysis of a harmful term |
| creative fiction | depiction within a narrative |

Moderation models can cause harm through false negatives and false positives. A false negative leaves harmful material unaddressed. A false positive suppresses legitimate expression, and these errors may be concentrated on dialects or identity discussions. Evaluation should report target-aware and context-aware slices rather than only one toxicity F1 score.

Generation systems need both input and output controls. A harmless-looking request can produce harmful content after several turns, while an input containing violent language may be a request for crisis support or academic analysis. Policy classification should use conversation context and intended transformation.

#### **Misinformation and Disinformation**

Misinformation is false or misleading information shared without necessarily intending deception. Disinformation is deliberately created or distributed to deceive. A model observing text alone often cannot determine intent, so system labels should avoid claiming more than the available evidence supports.

NLP systems participate in the information ecosystem in several ways:

```text
generate a claim
-> rewrite it persuasively
-> translate it across languages
-> personalize it for an audience
-> rank or recommend it
-> summarize reactions to it
```

Scale and automation can turn a modest model error into broad exposure. Conversely, automated fact-checking can also create false confidence when evidence is incomplete or contested.

Useful controls include source provenance, freshness checks, claim-level retrieval, authoritative-source prioritization, uncertainty, visible citation, rate limits, and human editorial review. For time-sensitive claims, the system should record when evidence was retrieved. For contested topics, it should distinguish verified fact, reported claim, expert consensus, uncertainty, and opinion.

Misinformation detection should not be reduced to classifying publishers or writing style. Reliable sources can make mistakes, satire can look false, and low-quality sources can quote true facts. The unit of verification should be the claim and its evidence whenever possible.

#### **Dangerous Advice and Dual-Use Capabilities**

Dual-use capability means the same NLP function can support beneficial or harmful goals. Translation can improve access or coordinate abuse. Code generation can teach programming or help exploit systems. Scientific summarization can support research or accelerate harmful experimentation.

Risk depends on capability, intent, detail, and actionability:

| Response Level | Example | Typical Handling |
|---|---|---|
| general information | high-level explanation of cybersecurity | usually provide |
| protective guidance | how to secure an account | provide concrete defensive steps |
| ambiguous operational detail | procedure could be used safely or harmfully | clarify context and narrow scope |
| high-risk actionability | optimized instructions for causing harm | restrict and redirect |
| urgent personal danger | credible self-harm or violence signal | supportive response and appropriate escalation path |

A safe response is not always a refusal. It may provide high-level educational context, defensive alternatives, emergency resources, uncertainty, or a recommendation to consult a qualified professional. The response should avoid presenting generated medical, legal, or financial information as a personalized professional decision.

#### **Over-Refusal and Under-Refusal**

**Under-refusal** occurs when a system provides content that should have been restricted. **Over-refusal** occurs when it blocks benign, educational, creative, or protective requests. Both are safety failures.

Examples of over-refusal include rejecting a historian's analysis of extremist propaganda, blocking a security engineer asking about defense, or refusing to translate a patient's description of symptoms. Over-refusal can reduce accessibility and disproportionately affect users whose language resembles policy examples.

Examples of under-refusal include giving detailed harmful instructions after superficial reframing, leaking sensitive content through a transformation request, or allowing a long conversation to assemble a disallowed plan in pieces.

Evaluation needs a balanced test set containing:

```text
clearly allowed requests
clearly disallowed requests
allowed requests with sensitive vocabulary
ambiguous requests requiring clarification
multi-turn escalation attempts
multilingual and encoded variants
```

The best behavior can be `comply`, `comply with boundaries`, `clarify`, `refuse with a safe alternative`, or `escalate`. Binary allow/block labels are often too coarse.

#### **Safety and Utility Trade-Offs**

Safety controls change precision and recall. Lowering a moderation threshold catches more harmful content but also flags more benign content. The appropriate threshold depends on error costs, reversibility, review capacity, and context.

Let $C_{FN}$ be the cost of a harmful false negative and $C_{FP}$ the cost of a benign false positive. A simplified expected cost is:

$$
C = C_{FN}N_{FN} + C_{FP}N_{FP}
$$

$N_{FN}$ and $N_{FP}$ are the observed counts of false negatives and false positives. The formula makes the trade-off explicit, but assigning costs is a policy judgment. It must not erase rights, rare severe outcomes, or unequal burdens across groups.

Tiered routing is often better than one threshold:

```text
low risk -> normal response
medium risk -> constrained response or clarification
high uncertainty -> human review
high risk -> block action and offer safe alternatives
immediate danger -> specialized escalation procedure
```

<details>
<summary>Python Policy-Based Safety Routing</summary>

```python
def route_request(harm_probability, uncertainty, high_impact_domain=False):
    """Return an action, not a final content judgment."""

    # Step 1: high-impact domains receive stricter review because an
    # incorrect answer can have larger consequences.
    review_threshold = 0.35 if high_impact_domain else 0.55
    restrict_threshold = 0.80

    # Step 2: high uncertainty is a reason to ask for review or context,
    # not a reason to confidently label the user as malicious.
    if uncertainty >= 0.40:
        return "clarify_or_human_review"

    # Step 3: route by risk tier. A production policy would also examine
    # category, target, conversation history, user permissions, and tools.
    if harm_probability >= restrict_threshold:
        return "restrict_and_offer_safe_alternative"
    if harm_probability >= review_threshold:
        return "constrained_response_or_review"
    return "normal_response"

cases = [
    {"name": "general education", "risk": 0.08, "uncertainty": 0.05},
    {"name": "ambiguous medical request", "risk": 0.30, "uncertainty": 0.45,
     "high_impact": True},
    {"name": "highly actionable harm", "risk": 0.93, "uncertainty": 0.08},
]

for case in cases:
    action = route_request(
        case["risk"],
        case["uncertainty"],
        case.get("high_impact", False),
    )
    print(case["name"], "->", action)
```

</details>

The code illustrates separation between detection and response policy. A score should not directly trigger an irreversible action without considering context, uncertainty, subgroup performance, and appeal. Safety is best understood as calibrated risk management rather than maximum refusal.

### **Transparency, Accountability, and Human Oversight**

Transparency makes relevant information about a system available to the people who need it. Accountability assigns responsibility for decisions and consequences. Human oversight creates meaningful opportunities for people to review, challenge, correct, pause, or override the system.

These concepts support one another, but documentation alone does not create accountability. A perfectly documented harmful system is still harmful. Likewise, adding a human reviewer does not help if the reviewer lacks information, time, authority, or a realistic alternative to accepting the model output.

#### **Interpretability and Explainability**

Interpretability concerns how understandable a model's behavior or internal structure is. Explainability concerns methods that produce reasons or evidence for a particular behavior. In practice, the terms are often used broadly, so the intended audience and purpose should be stated.

Different users need different explanations:

| Audience | Useful Explanation |
|---|---|
| developer | influential tokens, error slices, internal diagnostics |
| domain expert | evidence, uncertainty, and decision-relevant factors |
| decision subject | understandable reason and correction or appeal path |
| auditor | reproducible logs, versions, tests, and control evidence |
| executive or regulator | intended use, risk, ownership, and performance limits |

Local methods explain one prediction; global methods describe broader model behavior. Feature attribution, counterfactual examples, prototypes, concept tests, probing, and mechanistic analysis provide different evidence. Attention weights can be diagnostically useful but should not automatically be presented as a complete causal explanation.

Explanation quality has several dimensions:

| Dimension | Question |
|---|---|
| fidelity | does the explanation reflect the real decision process? |
| comprehensibility | can the audience understand it? |
| stability | do similar inputs receive similar explanations? |
| actionability | can the user correct or respond to the decision? |
| completeness | are uncertainty and limitations included? |

A fluent natural-language rationale may sound persuasive while being generated after the prediction and not causally connected to it. Explanations should therefore be validated rather than trusted because they are readable.

#### **Dataset, Model, and System Documentation**

Documentation preserves assumptions that would otherwise disappear when teams, data, models, or deployment conditions change. Different artifacts describe different objects:

| Artifact | Documents | Important Content |
|---|---|---|
| data card or datasheet | dataset | source, consent, population, labels, gaps, transformations |
| model card | trained model | intended use, evaluation, subgroup results, limitations |
| system card | deployed application | components, interactions, safety tests, access, controls |
| decision log | design choice | alternatives, evidence, owner, date, rationale |
| risk register | identified risk | likelihood, impact, control, status, owner |
| incident report | observed failure | timeline, effect, root cause, remediation |

Model cards were proposed to report intended uses and performance across relevant conditions and subgroups. They are most useful when concrete and versioned, not when filled with generic statements such as `may contain bias`.

A good model card for an NLP classifier should state label definitions, languages, domains, data period, threshold, slice metrics, known failure cases, calibration, privacy considerations, and out-of-scope uses. A RAG system additionally needs document provenance, index version, retrieval metrics, citation checks, and tool permissions.

Documentation should be proportional to impact but begin early. Writing limitations after deployment encourages teams to rationalize decisions that are already difficult to reverse.

#### **Auditability and Reproducibility**

Auditability means that an authorized reviewer can reconstruct what happened and evaluate whether controls were followed. Reproducibility means that a process or result can be repeated closely enough to validate the claim.

For modern NLP systems, record more than the model name:

```text
model and tokenizer version
-> code and configuration
-> data or index version
-> prompt and policy version
-> retrieval results and tool calls
-> decoding parameters and random seed where applicable
-> output filters and human actions
-> timestamp and deployment environment
```

Generative outputs can vary because of sampling, nondeterministic hardware, changing hosted models, external search results, or updated indexes. Exact reproduction may not always be possible, but the system should preserve enough evidence to explain the execution path.

Logging introduces privacy and security risk. Storing every prompt may capture secrets; storing raw tool outputs may expose credentials or personal records. Audit logs should minimize content, separate access roles, protect integrity, define retention, and record necessary provenance without becoming a second ungoverned dataset.

Independent evaluation is stronger when evaluators can inspect assumptions, test artifacts, and known incidents. Auditability is not the same as publishing all model weights or private data. Access can be controlled while still supporting meaningful oversight.

#### **Human-in-the-Loop Decision Making**

Human-in-the-loop is meaningful only when the human has a defined role. A person may label data, review uncertain cases, authorize an action, monitor patterns, hear appeals, or investigate incidents. These responsibilities require different skills and interfaces.

Three configurations are useful:

| Configuration | System Role | Human Role |
|---|---|---|
| human-in-the-loop | model proposes, human approves each consequential action | active decision maker |
| human-on-the-loop | system acts within limits, human monitors and can intervene | supervisor |
| human-out-of-the-loop | system acts automatically | governance relies on pre-deployment controls and monitoring |

Consequential decisions should specify who has authority to override the model, what evidence is shown, how long the reviewer has, and what happens when the reviewer disagrees. An appeal process should be available to the person affected, not only the operator.

Human review has limits. Reviewers can experience fatigue, inconsistent judgment, domain gaps, time pressure, and automation bias. If the system sends only the hardest cases to humans, their workload will be more difficult than average and performance assumptions based on ordinary cases will be misleading.

#### **Automation Bias and Limits of Human Oversight**

Automation bias is the tendency to over-rely on automated suggestions, especially when the system usually appears competent. People may accept a wrong answer, fail to search for contradictory evidence, or change their judgment to match the model.

Interfaces can unintentionally increase this bias by showing a precise score without uncertainty, placing the model recommendation first, hiding source evidence, or making override cumbersome. A nominal human approval step can become rubber-stamping when throughput targets leave seconds per case.

Useful countermeasures include:

| Control | Why It Helps |
|---|---|
| show source evidence before recommendation | encourages independent assessment |
| display calibrated uncertainty and known limitations | reduces false precision |
| require a reason for high-impact approval or override | supports deliberate review and audit |
| sample apparently easy cases | measures hidden automation errors |
| rotate tasks and manage workload | reduces fatigue |
| provide authority and escalation | makes intervention real |
| measure human-plus-model performance | evaluates the actual deployed team |

<details>
<summary>Python Creating a Minimal Versioned Model Card Record</summary>

```python
from datetime import datetime, timezone
import json

model_card = {
    "model_id": "support-urgency-classifier",
    "version": "2.3.0",
    "created_at": datetime.now(timezone.utc).isoformat(),
    "intended_use": "route English customer-support messages for review",
    "out_of_scope": [
        "medical emergency triage",
        "employee performance decisions",
        "languages not covered by evaluation",
    ],
    "decision_threshold": 0.72,
    "evaluation": {
        "data_period": "2026-Q1",
        "overall_macro_f1": 0.88,
        "required_slices": ["negation", "code_switching", "short_messages"],
    },
    "known_limitations": [
        "reduced recall on indirect requests",
        "not calibrated for new product domains",
    ],
    "owners": {
        "model": "NLP team",
        "policy": "support operations",
        "incident_response": "service reliability team",
    },
}

# Step 1: serialize the exact documentation record with the release.
print(json.dumps(model_card, indent=2))

# Step 2: deployment automation can verify that required fields and
# evaluations exist before accepting this model version.
required = {"model_id", "version", "intended_use", "evaluation", "owners"}
missing = required - model_card.keys()
assert not missing, f"model card is missing: {sorted(missing)}"
```

</details>

The record is intentionally small. Real documentation should link to data provenance, evaluation reports, risk decisions, and change history. The key is that documentation is part of release evidence rather than an informal page that can drift away from the deployed system.

### **Responsible NLP Lifecycle**

Responsible NLP is a continuous lifecycle rather than a final safety test. Risks can enter during problem definition, data collection, model development, integration, deployment, and user interaction. Controls must therefore be connected across the entire system.

The NIST AI Risk Management Framework organizes this work around four functions: **Govern**, **Map**, **Measure**, and **Manage**. Governance is cross-cutting, while mapping, measuring, and managing repeat as the system and context change.

> Image: the NIST AI RMF core functions.
>
> ![NIST AI risk management functions](assets/nist-ai-risk-management.png){width=75%}
>
> Source: [NIST AI Risk Management Framework 1.0](https://doi.org/10.6028/NIST.AI.100-1)

This framework is useful because it prevents teams from jumping directly to metrics. Before measuring a risk, the team must map the context, stakeholders, intended purpose, foreseeable misuse, and possible impacts. Measurement then informs concrete management decisions.

#### **Risk Assessment Before Development**

The first question is whether an NLP system should be built for the proposed decision. An AI solution may be unnecessary, less reliable than a simple rule, or inappropriate because the target cannot be validly inferred from language.

Pre-development assessment should define:

| Item | Question |
|---|---|
| purpose | what decision or user need is being supported? |
| non-AI baseline | can rules, search, process change, or human service solve it better? |
| stakeholders | who uses, is represented in, and is affected by the system? |
| intended use | where and by whom should it be used? |
| out-of-scope use | where must it not be used? |
| impact | what happens after a false positive or false negative? |
| data justification | why is each data source necessary and permitted? |
| success criteria | what quality, fairness, privacy, safety, and cost thresholds apply? |
| remedy | how can a person correct data or contest an outcome? |

This stage should identify assumptions that can later be tested. For example, a project may assume that support messages contain enough evidence to infer urgency, that the label policy is stable, and that users write in English. Each assumption becomes a risk if deployment violates it.

High-impact projects benefit from an impact assessment and multidisciplinary review before expensive model development begins. Early cancellation or scope reduction can be a successful risk-management outcome.

#### **Safety Evaluation Before Deployment**

Pre-deployment evaluation should test the complete system under realistic conditions. A model score from a static dataset does not test retrieval, prompts, tool permissions, user interface, moderation, human review, or failure recovery.

A release gate can require evidence across several dimensions:

```text
task quality
+ slice and fairness evaluation
+ robustness and adversarial tests
+ privacy and extraction tests
+ harmful-content and misuse tests
+ human workflow evaluation
+ latency, availability, and rollback tests
```

The test set should include normal cases, difficult linguistic cases, high-impact failures, out-of-scope requests, malicious inputs, and interactions across multiple turns. Human evaluators need clear rubrics and disagreement analysis. Red-team findings should be retested after remediation.

Release decisions should not depend on one composite score that allows strong accuracy to cancel a critical privacy failure. Some controls are hard gates: exposed secrets, unauthorized actions, or unacceptable subgroup failures may block deployment regardless of average utility.

#### **Monitoring After Deployment**

Deployment changes the evidence available. Real users produce new language, combine features unexpectedly, and reveal impacts absent from a benchmark. Monitoring should cover model behavior, system behavior, and real-world outcomes.

| Monitoring Area | Example Signal |
|---|---|
| input drift | new vocabulary, language, topic, or document length |
| output quality | sampled human ratings and correction rate |
| slice performance | subgroup false-negative rate or dialect error |
| grounding | unsupported claim and citation mismatch rate |
| safety | harmful completion, refusal, and escalation rate |
| privacy | PII detector alerts and unusual repeated queries |
| security | injection attempts and unauthorized tool requests |
| operations | latency, failure, cost, and fallback rate |
| impact | complaints, appeals, overturned decisions, and downstream harm |

Metrics need thresholds, owners, and response procedures. An alert without an owner is only a notification. Monitoring should include data needed to investigate while respecting privacy and retention limits.

User reports are an important signal but not a representative sample. People with less power or lower access may be less able to report harm. Proactive audits and community engagement are needed alongside complaint channels.

#### **Incident Response and Continuous Auditing**

An incident is an event in which the system causes or creates a credible risk of unacceptable harm, policy violation, privacy exposure, security compromise, or major reliability failure. Teams should define incident categories and response authority before an event occurs.

A practical response sequence is:

```text
detect
-> contain
-> preserve evidence
-> assess affected people and systems
-> communicate and provide remedy
-> identify root causes
-> remediate and validate
-> document lessons and update controls
```

Containment may disable a feature, revoke tool permissions, roll back a model, quarantine an index, tighten thresholds, or route all cases to human review. The action should reduce harm without destroying evidence needed for investigation.

Root-cause analysis should examine more than the final model prediction. A harmful outcome may involve ambiguous policy, missing provenance, weak interface design, workload pressure, inaccessible appeals, or a monitoring gap. Corrective actions should address each contributing layer.

Continuous auditing periodically checks whether documented controls still operate, whether deployment remains within scope, and whether new capabilities create new risks. Independent review is especially valuable where teams have incentives to interpret their own results optimistically.

#### **Practical Risk-Mitigation Workflow**

The complete workflow connects technical evidence with responsibility and action:

```text
1. define purpose and boundaries
2. map stakeholders, data, threats, and possible harms
3. choose measurable requirements and hard safety gates
4. build the least risky system that can meet the need
5. evaluate components and the end-to-end workflow
6. document residual risk and obtain accountable approval
7. deploy gradually with monitoring, fallback, and rollback
8. investigate incidents and user feedback
9. update, restrict, or retire the system when assumptions fail
```

<details>
<summary>Python Release Gate for a Responsible NLP System</summary>

```python
release_evidence = {
    "task_macro_f1": 0.89,
    "worst_slice_f1": 0.82,
    "max_equal_opportunity_gap": 0.04,
    "unsupported_claim_rate": 0.015,
    "critical_privacy_findings": 0,
    "critical_security_findings": 0,
    "rollback_test_passed": True,
    "human_review_ready": True,
}

requirements = {
    "task_macro_f1": (">=", 0.85),
    "worst_slice_f1": (">=", 0.78),
    "max_equal_opportunity_gap": ("<=", 0.08),
    "unsupported_claim_rate": ("<=", 0.02),
}

def meets(value, operator, threshold):
    if operator == ">=":
        return value >= threshold
    if operator == "<=":
        return value <= threshold
    raise ValueError(f"unsupported operator: {operator}")

failures = []

# Step 1: evaluate quantitative requirements independently.
for metric, (operator, threshold) in requirements.items():
    value = release_evidence[metric]
    if not meets(value, operator, threshold):
        failures.append(f"{metric} is {value}; requires {operator} {threshold}")

# Step 2: enforce hard gates that cannot be averaged away by good accuracy.
hard_gates = {
    "critical_privacy_findings": release_evidence["critical_privacy_findings"] == 0,
    "critical_security_findings": release_evidence["critical_security_findings"] == 0,
    "rollback_test_passed": release_evidence["rollback_test_passed"],
    "human_review_ready": release_evidence["human_review_ready"],
}

for gate, passed in hard_gates.items():
    if not passed:
        failures.append(f"hard gate failed: {gate}")

# Step 3: produce an auditable decision and preserve the evidence version.
decision = "GO" if not failures else "NO-GO"
print("release decision:", decision)
for failure in failures:
    print(" -", failure)
```

</details>

The thresholds are examples, not universal standards. They should be justified by domain risk, baseline performance, uncertainty, affected stakeholders, and available controls. A release gate supports accountable judgment; it does not replace it.

The chapter can be summarized as a set of connected questions:

| Area | Failure to Look For | Evidence | Main Response |
|---|---|---|---|
| language and domain | ambiguity, dialect, low-resource, and shift failures | contrast sets and deployment slices | contextual modeling and representative evaluation |
| factual reliability | unsupported or incorrect claims | claim-level evidence checks | grounding, abstention, verification, review |
| fairness | unequal representation, errors, or outcomes | subgroup and intersectional analysis | redesign, data, model, policy, and appeal |
| privacy | disclosure, inference, or memorization | data inventory and privacy tests | minimization, access, DP, retention, deletion |
| security | manipulation, poisoning, extraction, or injection | threat models and adaptive red-teaming | least privilege, validation, containment, monitoring |
| harmful use | toxic, deceptive, or dangerous behavior | contextual policy evaluation | tiered response, safe alternatives, escalation |
| transparency | missing evidence or opaque decisions | cards, logs, explanations, audits | documentation and traceability |
| human oversight | rubber-stamping or absent remedy | human-plus-model evaluation | authority, workload design, appeal |
| lifecycle governance | controls decay after release | monitoring, incidents, periodic audit | govern, map, measure, manage, and retire |

Responsible NLP does not mean eliminating all uncertainty or all failure. It means making assumptions explicit, measuring what matters, reducing preventable harm, preserving human agency, and maintaining the ability to detect, contain, correct, and learn from failures throughout the system lifecycle.

